# DAMICORE analyzes DAMICORE

DAMICORE was introduced as **Data-Mining of Code Repositories**: raw code can be treated as a collection of objects, compared with Normalized Compression Distance (NCD), arranged by Neighbor Joining, and partitioned into communities. This notebook applies that idea to the current implementation itself.

**Question:** what similarity structure does DamiCore recover from its own production Python modules?

This is an exploratory structural view, not an architecture-quality score. A shared cluster means that the chosen representation contains compressible similarity; it does not imply that modules should be merged or that package boundaries are wrong.

## Experimental contract

One object is one tracked Python **implementation module** from the five public DamiCore distributions.

- `packages/*/src/**/*.py` is considered only for the public DamiCore packages.
- `__init__.py` is excluded because it mainly declares package surfaces rather than implementation behavior.
- Tests, notebooks, documentation, generated files, and the private `synthetic_data` package are excluded.
- Files are analyzed as raw bytes. There is no tokenization, AST normalization, or language-specific feature extraction.
- The notebook uses the exact public pipeline: NCD → deterministic Neighbor Joining → FastGreedy clustering.
- A single worker keeps notebook execution simple and avoids process-pool behavior; it does not change the semantic result.

This notebook is self-contained: it does not assume it is running from inside a DamiCore checkout. It installs the published `damicore` package, and `matplotlib` with `pip`, and clones the source tree it analyzes directly from the `delbem-research/damicore` GitHub repository into a temporary directory, and prints the commit it analyzed so a saved copy of the notebook names the source it describes. It therefore needs network access to PyPI and GitHub, and `git` available on `PATH`.

Setting `DAMICORE_SELF_ANALYSIS_REPO` to an existing checkout skips the clone and analyzes that tree instead, which is how the repository's notebook test executes these cells with no network. A checkout supplied that way is read, never removed.

In [ ]:
%pip install --quiet damicore matplotlib

In [ ]:
import json
import os
import shutil
import subprocess
import tempfile
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from damicore import ExecutionConfig, estimate, run
from IPython.display import display
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter

REPO_URL = "https://github.com/delbem-research/damicore.git"
REPO_REF = "main"

PUBLIC_PACKAGES = {
    "damicore",
    "damicore_normalizer",
    "damicore_distance",
    "damicore_tree_builder",
    "damicore_clusterizer",
}

# One color per source package, shared by every figure below.
PACKAGE_COLORS = {
    "damicore": "tab:blue",
    "damicore_normalizer": "tab:orange",
    "damicore_distance": "tab:green",
    "damicore_tree_builder": "tab:purple",
    "damicore_clusterizer": "tab:pink",
}

# A checkout to analyze. DAMICORE_SELF_ANALYSIS_REPO names one that already exists, which is
# how this repository's notebook test executes these cells with no network; without it the
# notebook clones, which is what a Colab reader wants.
existing = os.environ.get("DAMICORE_SELF_ANALYSIS_REPO")
cloned = not existing
if existing:
    repo_root = Path(existing).resolve()
    origin = f"local checkout: {repo_root}"
else:
    repo_root = Path(tempfile.mkdtemp(prefix="damicore-self-analysis-repo-"))
    subprocess.run(
        [
            "git",
            "clone",
            "--quiet",
            "--depth",
            "1",
            "--branch",
            REPO_REF,
            REPO_URL,
            str(repo_root),
        ],
        check=True,
    )
    origin = f"cloned repository ({REPO_URL}@{REPO_REF}): {repo_root}"

tracked = subprocess.check_output(
    ["git", "ls-files", "packages"],
    cwd=repo_root,
    text=True,
).splitlines()

analyzed_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo_root, text=True
).strip()

relative_modules = []
for relative in tracked:
    path = Path(relative)
    if (
        len(path.parts) >= 5
        and path.parts[0] == "packages"
        and path.parts[1] in PUBLIC_PACKAGES
        and "src" in path.parts
        and path.suffix == ".py"
        and path.name != "__init__.py"
    ):
        relative_modules.append(path)

relative_modules.sort(key=lambda path: path.as_posix())
corpus = [repo_root / path for path in relative_modules]

if len(corpus) < 2:
    raise RuntimeError("Self-analysis requires at least two tracked production modules.")

corpus_summary = pd.DataFrame(
    {
        "package": [path.parts[1] for path in relative_modules],
        "module": [path.as_posix() for path in relative_modules],
        "bytes": [absolute.stat().st_size for absolute in corpus],
    }
)

print(origin)
print(f"analyzed commit: {analyzed_commit}")
display(
    corpus_summary.groupby("package", sort=True)
    .agg(modules=("module", "count"), bytes=("bytes", "sum"))
    .sort_index()
)


# Labels are paths relative to packages/, so the first part names the package.
def package_of(label):
    return Path(label).parts[0]


def short_label(label):
    """`distance/ncd` for `damicore_distance/src/damicore_distance/ncd.py`."""
    path = Path(label)
    return (
        Path(path.parts[0].removeprefix("damicore_")) / Path(*path.parts[3:]).with_suffix("")
    ).as_posix()


def color_ticks(ticks, labels):
    for tick, label in zip(ticks, labels):
        tick.set_color(PACKAGE_COLORS[package_of(label)])
        tick.set_fontsize(8)

## Corpus balance

The amount of code contributed by each package affects how much evidence that package contributes to the analysis. These views separate object count from total source bytes: a package can have many small modules or a few large ones. They are descriptive context, not weights applied to NCD.

In [ ]:
package_summary = corpus_summary.groupby("package", sort=True).agg(
    modules=("module", "count"), bytes=("bytes", "sum")
)
package_names = [package.removeprefix("damicore_") for package in package_summary.index]
bar_colors = [PACKAGE_COLORS[package] for package in package_summary.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(package_names, package_summary["modules"], color=bar_colors)
axes[0].set_title("Implementation modules per package")
axes[0].set_ylabel("module count")
axes[1].bar(package_names, package_summary["bytes"] / 1024, color=bar_colors)
axes[1].set_title("Raw Python source per package")
axes[1].set_ylabel("KiB")
for axis in axes:
    axis.set_xlabel("package")
    axis.tick_params(axis="x", rotation=25)
    axis.grid(axis="y", alpha=0.25)
fig.suptitle("Corpus balance", fontsize=15, fontweight="bold")
fig.tight_layout()
plt.show()

## Pre-flight

DamiCore's cost is driven primarily by the number of objects: NCD is quadratic in object count and Neighbor Joining is cubic. `estimate()` performs the same input scan used by the run and decides whether the exact analysis fits the configured resource gates before any run artifacts are created.

In [ ]:
execution = ExecutionConfig(workers=1)
preview = estimate(
    corpus,
    source_kind="files",
    execution=execution,
)

mib = 1024**2
display(
    pd.Series(
        {
            "objects": preview.object_count,
            "pairs": preview.pair_count,
            "input MiB": preview.input_size_bytes / mib,
            "matrix MiB": preview.matrix_bytes / mib,
            "estimated working memory MiB": preview.estimated_working_memory_bytes / mib,
            "required free disk MiB": preview.required_free_disk_bytes / mib,
            "within limits": preview.within_limits,
        },
        name="pre-flight",
    ).to_frame()
)

if not preview.within_limits:
    raise RuntimeError(f"Pre-flight rejected the run: {preview.violations}")

## Run the exact pipeline

The run uses the same corpus and execution configuration that were pre-flighted. Artifacts go to a fresh temporary directory so the repository checkout is not modified.

In [ ]:
run_dir = Path(tempfile.mkdtemp(prefix="damicore-self-analysis-"))

result = run(
    corpus,
    source_kind="files",
    output_dir=run_dir,
    progress=False,
    execution=execution,
)

display(
    pd.Series(
        {
            "status": result.report.status,
            "objects": result.report.object_count,
            "clusters": result.report.cluster_count,
            "modularity": result.report.modularity,
            "run directory": str(result.artifacts.run_dir),
        },
        name="result",
    ).to_frame()
)

## Distance heatmap

The matrix below shows the full pairwise NCD between every module, in package order. Darker cells are smaller distances, so dark blocks along the diagonal suggest modules with similar raw-byte structure. Labels are abbreviated to `package/module` and colored by source package; the membership table below keeps the complete paths.

In [ ]:
matrix = result.distance_matrix
labels = list(matrix.labels)
values = np.asarray(matrix[:, :], dtype=float)
packages = [package_of(label) for label in labels]


def plot_distance_map(order, title):
    ordered = [labels[index] for index in order]
    fig, ax = plt.subplots(figsize=(15, 12))
    image = ax.imshow(values[np.ix_(order, order)], cmap="viridis", aspect="auto")
    ax.set_title(title)
    ticks = [short_label(label) for label in ordered]
    ax.set_xticks(range(len(ticks)), ticks, rotation=90)
    ax.set_yticks(range(len(ticks)), ticks)
    color_ticks(ax.get_xticklabels(), ordered)
    color_ticks(ax.get_yticklabels(), ordered)
    fig.colorbar(image, ax=ax, label="NCD")
    return fig, ax


fig, ax = plot_distance_map(list(range(len(labels))), "Pairwise NCD between DAMICORE modules")
fig.tight_layout()
plt.show()

## Tree

The Neighbor Joining tree, drawn from the run's `tree.json`, is the structure the clustering partitions. Each leaf is a module; modules joined by shorter paths have lower NCD. Leaf colors show the source package, not the cluster, so the discovered structure can be compared with the known architecture.

In [ ]:
# tree.json is the run's own tree artifact: edges point from parent to child, and leaves are
# object ids that the membership table maps back to module paths.
tree_graph = json.loads((result.artifacts.run_dir / "tree.json").read_text(encoding="utf-8"))
label_of = dict(zip(result.membership["object_id"], result.membership["label"]))
children = defaultdict(list)
for edge in tree_graph["edges"]:
    children[edge["source"]].append((edge["target"], edge["length"]))

# Rectangular layout: x is the branch length accumulated from the root, each leaf takes the
# next row, and an internal node sits midway between its outermost children.
x_of, y_of, leaves = {}, {}, []


def place(node, x):
    x_of[node] = x
    if node not in children:
        y_of[node] = len(leaves)
        leaves.append(node)
    else:
        rows = [place(child, x + length) for child, length in children[node]]
        y_of[node] = (min(rows) + max(rows)) / 2
    return y_of[node]


place(tree_graph["root_id"], 0.0)

fig, ax = plt.subplots(figsize=(17, max(11, 0.34 * len(leaves))))
for parent, edges in children.items():
    rows = [y_of[child] for child, _ in edges]
    ax.plot([x_of[parent]] * 2, [min(rows), max(rows)], color="#555555", linewidth=1)
    for child, _ in edges:
        ax.plot([x_of[parent], x_of[child]], [y_of[child]] * 2, color="#555555", linewidth=1)
for leaf in leaves:
    label = label_of[leaf]
    ax.text(
        x_of[leaf],
        y_of[leaf],
        f" {short_label(label)}",
        va="center",
        fontsize=8,
        color=PACKAGE_COLORS[package_of(label)],
    )
ax.set_yticks([])
ax.invert_yaxis()
ax.set_xlim(0, max(x_of.values()) * 1.35)
ax.set_title("Similarity structure of DAMICORE modules", fontsize=17, fontweight="bold")
ax.set_xlabel("accumulated branch length (NCD)")
ax.set_ylabel(f"modules ({len(leaves)} objects)")
ax.legend(
    handles=[
        Patch(color=color, label=package.removeprefix("damicore_"))
        for package, color in PACKAGE_COLORS.items()
    ],
    title="Source package",
    loc="lower right",
)
fig.tight_layout()
plt.show()

## Do discovered communities resemble package boundaries?

Package ownership is known independently from the clustering, so it is a useful external label for reading the result. Agreement is interesting, but disagreement is not automatically a defect: DamiCore measures compressible similarity in raw module bytes, while package boundaries encode software responsibilities.

In [ ]:
membership = result.membership.copy()
membership["package"] = membership["label"].map(lambda label: Path(label).parts[0])

display(
    membership[["cluster", "package", "label"]]
    .sort_values(["cluster", "package", "label"])
    .reset_index(drop=True)
)

display(
    pd.crosstab(
        membership["cluster"],
        membership["package"],
        rownames=["cluster"],
        colnames=["package"],
    )
)

## Cluster-ordered distance map

The same matrix reordered by discovered cluster asks a different question: whether FastGreedy communities occupy compact regions of the underlying distances. White lines mark cluster boundaries; label colors still show package ownership.

In [ ]:
cluster_of = membership.set_index("label")["cluster"].to_dict()
order = sorted(range(len(labels)), key=lambda index: (cluster_of[labels[index]], labels[index]))
fig, ax = plot_distance_map(order, "Pairwise NCD ordered by discovered cluster")
clusters_in_order = [cluster_of[labels[index]] for index in order]
for position in range(1, len(order)):
    if clusters_in_order[position] != clusters_in_order[position - 1]:
        ax.axhline(position - 0.5, color="white", linewidth=1.2)
        ax.axvline(position - 0.5, color="white", linewidth=1.2)
fig.tight_layout()
plt.show()

## Cluster composition

Raw counts reveal cluster size; proportions reveal purity independently of size. Read together, the panels show whether a community is dominated by one package or connects responsibilities implemented in several packages. Mixed clusters are structural observations, not architecture violations.

In [ ]:
counts = pd.crosstab(membership["cluster"], membership["package"]).reindex(
    columns=sorted(PUBLIC_PACKAGES), fill_value=0
)
shares = counts.div(counts.sum(axis=1), axis=0)
colors = [PACKAGE_COLORS[package] for package in counts.columns]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
counts.plot.barh(stacked=True, color=colors, ax=axes[0], legend=False)
axes[0].set_title("Modules in each cluster")
axes[0].set_xlabel("module count")
shares.plot.barh(stacked=True, color=colors, ax=axes[1], legend=False)
axes[1].set_title("Package share within each cluster")
axes[1].set_xlabel("share of cluster")
axes[1].set_xlim(0, 1)
axes[1].xaxis.set_major_formatter(PercentFormatter(1.0))
for axis in axes:
    axis.grid(axis="x", alpha=0.2)
fig.legend(
    [package.removeprefix("damicore_") for package in counts.columns],
    title="Source package",
    loc="lower center",
    ncol=len(counts.columns),
)
fig.suptitle("Discovered community composition", fontsize=15, fontweight="bold")
fig.tight_layout(rect=(0, 0.1, 1, 1))
plt.show()

## Which modules are closest?

Clusters summarize the global structure. The smallest off-diagonal NCD values expose the strongest pairwise similarities directly. The package-level table then compresses the same pairwise evidence into mean distances within and across packages.

In [ ]:
pair_rows = []
for left in range(len(labels)):
    for right in range(left + 1, len(labels)):
        package_a, package_b = sorted((packages[left], packages[right]))
        pair_rows.append(
            {
                "left": labels[left],
                "right": labels[right],
                "package_a": package_a,
                "package_b": package_b,
                "ncd": values[left, right],
            }
        )

pairs = pd.DataFrame(pair_rows).sort_values("ncd", ignore_index=True)

display(pairs[["left", "right", "ncd"]].head(15))

package_distance = pd.DataFrame(
    np.nan,
    index=sorted(PUBLIC_PACKAGES),
    columns=sorted(PUBLIC_PACKAGES),
)

for (package_a, package_b), group in pairs.groupby(["package_a", "package_b"]):
    mean_distance = group["ncd"].mean()
    package_distance.loc[package_a, package_b] = mean_distance
    package_distance.loc[package_b, package_a] = mean_distance

display(package_distance.round(3))

## Within-package and cross-package distances

If package boundaries align with raw-code similarity, within-package pairs should tend to have lower NCD than cross-package pairs. The histogram compares the full distributions (dashed lines are medians) rather than only their means; the annotated heatmap is the package-level table above.

In [ ]:
same_package = pairs["package_a"] == pairs["package_b"]
within_package = pairs.loc[same_package, "ncd"]
cross_package = pairs.loc[~same_package, "ncd"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
bins = np.linspace(pairs["ncd"].min(), pairs["ncd"].max(), 18)
for sample, name, color in (
    (within_package, "within package", "tab:blue"),
    (cross_package, "cross package", "tab:orange"),
):
    axes[0].hist(
        sample, bins=bins, density=True, alpha=0.6, color=color, label=f"{name} (n={len(sample)})"
    )
    axes[0].axvline(sample.median(), color=color, linestyle="--", linewidth=1.5)
axes[0].set_title("Distribution of pairwise distances")
axes[0].set_xlabel("NCD")
axes[0].set_ylabel("density")
axes[0].legend()

short_packages = [package.removeprefix("damicore_") for package in package_distance.index]
image = axes[1].imshow(package_distance.to_numpy(dtype=float), cmap="viridis")
axes[1].set_xticks(range(len(short_packages)), short_packages, rotation=35, ha="right")
axes[1].set_yticks(range(len(short_packages)), short_packages)
axes[1].set_title("Mean NCD by package pair")
for row in range(len(short_packages)):
    for column in range(len(short_packages)):
        value = package_distance.iloc[row, column]
        # A package with a single module has no within-package pair, so its cell is empty.
        if not np.isnan(value):
            axes[1].text(column, row, f"{value:.3f}", ha="center", va="center", color="white")
fig.colorbar(image, ax=axes[1], label="mean NCD", fraction=0.046, pad=0.04)
fig.suptitle(
    "How strongly do package boundaries structure distance?", fontsize=15, fontweight="bold"
)
fig.tight_layout()
plt.show()

## Nearest-neighbor bridges

Each bar is a module's closest peer: green stays within the package, red crosses a package boundary. This exposes both strong bridges and comparatively isolated modules across the whole corpus, not only the globally closest pairs.

In [ ]:
off_diagonal = values.copy()
np.fill_diagonal(off_diagonal, np.inf)
neighbor = off_diagonal.argmin(axis=1)
nearest = pd.DataFrame(
    {
        "module": [short_label(label) for label in labels],
        "neighbor": [short_label(labels[index]) for index in neighbor],
        "ncd": off_diagonal[np.arange(len(labels)), neighbor],
        "same_package": [packages[index] == packages[peer] for index, peer in enumerate(neighbor)],
    }
).sort_values("ncd", ascending=False, ignore_index=True)

fig, ax = plt.subplots(figsize=(15, max(10, 0.32 * len(nearest))))
ax.barh(
    range(len(nearest)),
    nearest["ncd"],
    color=nearest["same_package"].map({True: "tab:green", False: "tab:red"}),
)
ax.set_yticks(
    range(len(nearest)),
    [f"{row.module}  ->  {row.neighbor}" for row in nearest.itertuples()],
    fontsize=8,
)
ax.set_xlabel("nearest-neighbor NCD")
ax.set_title("Closest peer of every module")
ax.grid(axis="x", alpha=0.2)
ax.legend(
    handles=[
        Patch(color="tab:green", label="same package"),
        Patch(color="tab:red", label="cross package"),
    ],
    loc="lower right",
)
fig.tight_layout()
plt.show()

## Module size and average distance

NCD normalizes compressed size, but finite compressor overhead can still make very small files behave differently. Plotting source size against each module's mean distance is a diagnostic for that possible representation effect. A visible association would motivate a controlled follow-up; it would not by itself prove size bias or explain a cluster.

In [ ]:
module_bytes = np.array(
    [(repo_root / "packages" / label).stat().st_size for label in labels], dtype=float
)
mean_distance = (values.sum(axis=1) - np.diag(values)) / (len(labels) - 1)
log_size_correlation = float(np.corrcoef(np.log10(module_bytes), mean_distance)[0, 1])

fig, ax = plt.subplots(figsize=(11, 7))
for package in sorted(PUBLIC_PACKAGES):
    mask = np.array(packages) == package
    ax.scatter(
        module_bytes[mask],
        mean_distance[mask],
        color=PACKAGE_COLORS[package],
        label=package.removeprefix("damicore_"),
        s=65,
        alpha=0.85,
    )
# Label only the extrema so the diagnostic stays readable.
for index in sorted(
    {module_bytes.argmin(), module_bytes.argmax(), mean_distance.argmin(), mean_distance.argmax()}
):
    ax.annotate(
        short_label(labels[index]),
        (module_bytes[index], mean_distance[index]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )
ax.set_xscale("log")
ax.set_xlabel("raw module size (bytes, log scale)")
ax.set_ylabel("mean NCD to all other modules")
ax.set_title(f"Module size vs. average distance (log-size Pearson r = {log_size_correlation:.2f})")
ax.grid(alpha=0.2)
ax.legend(title="Source package")
fig.tight_layout()
plt.show()

## Reading the result

Three observations are worth separating:

1. **Within-package concentration** suggests that modules owned by the same distribution share substantial raw-code structure.
2. **Cross-package neighbors** identify implementation similarities that cut across the architecture and are good candidates for human inspection.
3. **Cluster/package disagreement** is evidence about this representation, not a refactoring instruction.

The original DAMICORE work showed that changing the representation of code can change the structure that becomes visible. This notebook deliberately uses only raw source bytes so the experiment stays minimal and the interpretation stays clear. A future experiment could compare this baseline with a separately defined structural representation, but that is a different scientific question.

In [ ]:
result.close()
# Only a checkout this notebook created is removed. One it was merely pointed at belongs to
# whoever pointed at it, and rmtree on a working tree would delete the repository itself.
if cloned:
    shutil.rmtree(repo_root, ignore_errors=True)
    print(f"Closed the result memory map and removed the cloned repository at: {repo_root}")
else:
    print(f"Closed the result memory map; left the checkout at {repo_root} untouched")
print(f"Run artifacts remain at: {run_dir}")